# Week 2 Day 2 (Rewritten for Gemini using LangGraph/LangChain)
We are going to build a simple Agent system for generating cold sales outreach emails using Gemini!
This notebook replaces the `openai-agents` SDK with standard `langchain` and `langchain_openai` which are fully compatible with Gemini via its OpenAI compatibility layer.


In [14]:
import os
import asyncio
from dotenv import load_dotenv

# Load environment variables
load_dotenv(r'c:\Users\abhin\Dropbox\PC\Downloads\projects\.env')

# Setup Groq as an OpenAI-compatible endpoint
os.environ["OPENAI_API_KEY"] = os.environ.get("GROQ_API_KEY", "")
os.environ["OPENAI_BASE_URL"] = "https://api.groq.com/openai/v1"

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="llama-3.3-70b-versatile")


In [15]:
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content

def send_test_email():
    api_key = os.environ.get('SENDGRID_API_KEY')
    if not api_key:
        print("No SendGrid API Key found!")
        return
    sg = sendgrid.SendGridAPIClient(api_key=api_key)
    from_email = Email("abhinavsingh649@gmail.com") 
    to_email = To("abhinavsingh649@gmail.com") 
    content = Content("text/plain", "This is a test email from Gemini!")
    try:
        mail = Mail(from_email, to_email, "Test email", content).get()
        response = sg.client.mail.send.post(request_body=mail)
        print("Email status code:", response.status_code)
    except Exception as e:
        print("Error:", e)

# send_test_email()


### Step 1: Create our Agents

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

instructions1 = "You are a professional, serious sales agent working for ComplAI. Write a cold email for the given request."
instructions2 = "You are a humorous, engaging sales agent working for ComplAI. Write a witty cold email."
instructions3 = "You are a busy, concise sales agent working for ComplAI. Write a short, to-the-point cold email."

def create_agent(instructions):
    prompt = ChatPromptTemplate.from_messages([
        ("system", instructions),
        ("user", "{input}")
    ])
    return prompt | llm | StrOutputParser()

agent1 = create_agent(instructions1)
agent2 = create_agent(instructions2)
agent3 = create_agent(instructions3)


### Step 2: Run in Parallel

In [17]:
async def generate_emails(message):
    results = await asyncio.gather(
        agent1.ainvoke({"input": message}),
        agent2.ainvoke({"input": message}),
        agent3.ainvoke({"input": message})
    )
    return results

emails = await generate_emails("Write a cold sales email addressed to 'Dear CEO'")
for i, email in enumerate(emails):
    print(f"\n--- Email {i+1} ---\n{email}")



--- Email 1 ---
Subject: Revolutionize Your Business with AI-Driven Solutions from ComplAI

Dear CEO,

I am reaching out to introduce ComplAI, a cutting-edge technology firm that specializes in developing innovative AI-driven solutions designed to transform the way businesses operate. Our team of experts has been at the forefront of artificial intelligence development, creating tailored solutions that drive growth, enhance efficiency, and reduce costs for forward-thinking companies like yours.

At ComplAI, we understand the challenges that come with navigating an ever-evolving business landscape. Our solutions are crafted to address these challenges head-on, providing your organization with the tools it needs to stay ahead of the competition. From streamlining operations and improving customer experiences to unlocking new revenue streams and gaining valuable insights, our AI-powered technologies can have a profound impact on your business's success.

Some of the key benefits our solut

### Step 3: Pick the Best Email

In [18]:
picker_prompt = ChatPromptTemplate.from_messages([
    ("system", "You pick the best cold sales email from the options. Imagine you are a customer and pick the one you are most likely to respond to. Reply ONLY with the exact text of the selected email. Do not give any explanation."),
    ("user", "Options:\n\n{emails}")
])
picker_agent = picker_prompt | llm | StrOutputParser()

combined_emails = "\n\n---NEXT OPTION---\n\n".join(emails)
best_email = await picker_agent.ainvoke({"emails": combined_emails})

print("\n*** BEST EMAIL ***\n")
print(best_email)



*** BEST EMAIL ***

Subject: Revolutionize Your Business with AI-Driven Solutions from ComplAI

Dear CEO,

I am reaching out to introduce ComplAI, a cutting-edge technology firm that specializes in developing innovative AI-driven solutions designed to transform the way businesses operate. Our team of experts has been at the forefront of artificial intelligence development, creating tailored solutions that drive growth, enhance efficiency, and reduce costs for forward-thinking companies like yours.

At ComplAI, we understand the challenges that come with navigating an ever-evolving business landscape. Our solutions are crafted to address these challenges head-on, providing your organization with the tools it needs to stay ahead of the competition. From streamlining operations and improving customer experiences to unlocking new revenue streams and gaining valuable insights, our AI-powered technologies can have a profound impact on your business's success.

Some of the key benefits our s

### Step 4: Use a Tool to send

In [19]:
from langchain_core.tools import tool

@tool
def send_email(body: str) -> str:
    """Send out an email with the given body to the sales prospect."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("abhinavsingh649@gmail.com") 
    to_email = To("abhinavsingh649@gmail.com") 
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    try:
        sg.client.mail.send.post(request_body=mail)
        return "Success"
    except Exception as e:
        return str(e)

llm_with_tools = llm.bind_tools([send_email])


In [20]:
manager_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Sales Manager at ComplAI. You have selected a winning email draft.\n"
               "Now you must use the send_email tool to send exactly the draft text to the prospect.\n"
               "Do not modify the text or add pleasantries."),
    ("user", "Please send the following email draft:\n\n{email_draft}")
])

manager_agent = manager_prompt | llm_with_tools

response = await manager_agent.ainvoke({"email_draft": best_email})

if response.tool_calls:
    print("Manager decided to call tool:", response.tool_calls[0]['name'])
    # Execute the tool
    tool_msg = send_email.invoke(response.tool_calls[0]['args'])
    print("Tool execution result:", tool_msg)
else:
    print("Manager didn't call the tool. Output was:", response.content)


Manager didn't call the tool. Output was: <function(send_email)>"body": "Subject: Revolutionize Your Business with AI-Driven Solutions from ComplAI Dear CEO, I am reaching out to introduce ComplAI, a cutting-edge technology firm that specializes in developing innovative AI-driven solutions designed to transform the way businesses operate. Our team of experts has been at the forefront of artificial intelligence development, creating tailored solutions that drive growth, enhance efficiency, and reduce costs for forward-thinking companies like yours. At ComplAI, we understand the challenges that come with navigating an ever-evolving business landscape. Our solutions are crafted to address these challenges head-on, providing your organization with the tools it needs to stay ahead of the competition. From streamlining operations and improving customer experiences to unlocking new revenue streams and gaining valuable insights, our AI-powered technologies can have a profound impact on your bu

### Handoffs (Passing control ACROSS)\n\nIn `openai-agents`, handoffs let one agent pass control completely to another. In LangChain/LangGraph, we achieve this by defining a workflow graph or explicitly routing the output of one agent to the input of the next.

In [24]:
# Let's create the specialized formatting agents
subject_prompt = ChatPromptTemplate.from_messages([
    ("system", "You write catchy subjects for cold sales emails. Reply ONLY with the subject line."),
    ("user", "{email_body}")
])
subject_writer = subject_prompt | llm | StrOutputParser()

html_prompt = ChatPromptTemplate.from_messages([
    ("system", "Convert the email body to simple, clean HTML. Use basic tags only (p, h1, h2, ul, li, a, br). Do not use inline styles or complex CSS. Keep it minimal and professional. Reply ONLY with the HTML code."),
    ("user", "{email_body}")
])
html_converter = html_prompt | llm | StrOutputParser()


In [27]:
# Fix: Override html_converter with simpler HTML generation that avoids problematic characters
html_prompt_simple = ChatPromptTemplate.from_messages([
    ("system", "Convert email body to clean HTML. Use ONLY basic tags: html, head, body, p, h1, h2, h3, ul, li, a, br. NO inline styles. NO complex CSS. NO backslashes. NO escaped quotes. Use single quotes in URLs if needed. Keep it minimal and professional. Reply ONLY with valid HTML code nothing else."),
    ("user", "{email_body}")
])
html_converter = html_prompt_simple | llm | StrOutputParser()

In [22]:
@tool
def send_html_email(subject: str, html_body: str) -> str:
    """Send out an email with the given subject and HTML body."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("abhinavsingh649@gmail.com")
    to_email = To("abhinavsingh649@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    try:
        sg.client.mail.send.post(request_body=mail)
        return "Success"
    except Exception as e:
        return str(e)

emailer_llm_with_tools = llm.bind_tools([send_html_email])


In [26]:
# The Email Manager handles the final leg of the journey
emailer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an Email Manager. You have received a final HTML body and a subject line.\n"
               "Your ONLY job is to use the send_html_email tool to send it out."),
    ("user", "Subject: {subject}\n\nHTML Body:\n{html_body}")
])
emailer_agent = emailer_prompt | emailer_llm_with_tools

async def execute_handoff_workflow(winning_email_text):
    print("1. Sales Manager hands off to formatting...")
    
    # Run formatting in parallel
    print("2. Generating Subject and HTML in parallel...")
    subject, html = await asyncio.gather(
        subject_writer.ainvoke({"email_body": winning_email_text}),
        html_converter.ainvoke({"email_body": winning_email_text})
    )
    
    print(f"\n[Subject Generated]: {subject}")
    
    # Handoff to Email Manager
    print("\n3. Handing off to Email Manager to send...")
    response = await emailer_agent.ainvoke({
        "subject": subject,
        "html_body": html
    })
    
    if response.tool_calls:
        print("Email Manager is calling tool:", response.tool_calls[0]['name'])
        result = send_html_email.invoke(response.tool_calls[0]['args'])
        print("Result:", result)
    else:
        print("Email Manager didn't call the tool.")


# Run the handoff workflow!
    return subject

subject = await execute_handoff_workflow(best_email)


1. Sales Manager hands off to formatting...
2. Generating Subject and HTML in parallel...

[Subject Generated]: Unlock the Full Potential of Your Business with AI-Driven Innovation

3. Handing off to Email Manager to send...


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=send_html_email>{"subject": "Unlock the Full Potential of Your Business with AI-Driven Innovation", "html_body": "<html><head></head><body><h1>Revolutionize Your Business with AI-Driven Solutions from ComplAI</h1><p>Dear CEO,</p><p>We are introducing ComplAI, a technology firm that specializes in developing AI-driven solutions to transform businesses. Our team of experts creates tailored solutions to drive growth, enhance efficiency, and reduce costs.</p><h2>About ComplAI</h2><p>At ComplAI, we understand the challenges of navigating the business landscape. Our solutions address these challenges, providing your organization with the necessary tools to stay ahead of the competition. From streamlining operations to unlocking new revenue streams,

### Phase 2: Dynamic Email Reply Handling

We will now connect to your Gmail inbox via IMAP to wait for an actual reply from the prospect! Once received, the `ReplyAgent` will draft and send a follow-up.
**Make sure you have `EMAIL_APP_PASSWORD` set in your `.env` file!**


In [ ]:
import asyncio
from imap_tools import MailBox, A

# Create the Reply Agent
reply_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Customer Success Agent. You are responding to a prospect's reply to our cold email. "
               "Be polite, concise, and address their concerns. Do not invent exact pricing, but say we can discuss it on a call."),
    ("user", "Original Email We Sent:\n{original_email}\n\nProspect's Reply:\n{prospect_reply}")
])
reply_agent = reply_prompt | llm | StrOutputParser()

async def wait_for_reply(expected_subject, check_interval=10):
    clean_subject = expected_subject.replace("Re: ", "").strip()
    print(f"Waiting for an unread reply matching subject: '{clean_subject}'...")
    
    email_user = "abhinavsingh649@gmail.com"
    email_pass = os.environ.get('EMAIL_APP_PASSWORD')
    
    if not email_pass:
        print("ERROR: Please set EMAIL_APP_PASSWORD in your .env file!")
        return None
        
    while True:
        try:
            # Login and check inbox
            with MailBox('imap.gmail.com').login(email_user, email_pass) as mailbox:
                # Search for unread emails
                for msg in mailbox.fetch(limit=10, reverse=True):
                    if clean_subject.lower() in msg.subject.lower() and msg.subject.lower().startswith("re:"):
                        print(f"\n✅ Found reply! Subject: {msg.subject}")
                        # Return the text body of the reply
                        return msg.text or msg.html
            
            # Print a dot to show it's polling
            print(".", end="", flush=True)
            await asyncio.sleep(check_interval)
        except Exception as e:
            print(f"\nIMAP Error: {e}")
            await asyncio.sleep(check_interval)

async def handle_dynamic_reply(original_email, subject_sent):
    print("1. Polling inbox for a reply...")
    prospect_reply = await wait_for_reply(subject_sent)
    
    if not prospect_reply:
        return
        
    print("\n2. Generating follow-up response...")
    
    # Generate the reply draft
    follow_up_draft = await reply_agent.ainvoke({
        "original_email": original_email,
        "prospect_reply": prospect_reply
    })
    
    print(f"\n[Follow-up Draft Generated]:\n{follow_up_draft}\n")
    
    # Hand off to formatting and sending!
    print("3. Handing off to formatting and Email Manager...")
    
    subject, html = await asyncio.gather(
        subject_writer.ainvoke({"email_body": follow_up_draft}),
        html_converter.ainvoke({"email_body": follow_up_draft})
    )
    
    print(f"\n[Follow-up Subject Generated]: {subject}")
    
    response = await emailer_agent.ainvoke({
        "subject": subject,
        "html_body": html
    })
    
    if response.tool_calls:
        print("Email Manager is calling tool:", response.tool_calls[0]['name'])
        result = send_html_email.invoke(response.tool_calls[0]['args'])
        print("Result:", result)
    else:
        print("Email Manager didn't call the tool.")

# Note: 'subject' is the variable generated during the Handoff phase for the HTML email
# If you run the whole notebook, 'subject' will be defined.
try:
    await handle_dynamic_reply(best_email, subject)
except NameError:
    print("Please run the Handoff cell above first to generate the initial 'subject' variable.")


Please run the Handoff cell above first to generate the initial 'subject' variable.
